# Student Score Prediction System Using Machine Learning
**Minor Project 1 | Beginner-friendly Google Colab notebook**

## Important data note
This starter project uses a **synthetic (artificially generated) dataset** so that the notebook runs immediately and reproducibly. It is not real student data. If your college requires a real/public dataset, replace this CSV with an approved dataset and update the report accordingly.

### Objective
Predict an estimated exam score using study hours, attendance, previous score, and sleep hours. This is a learning demonstration, not a reliable tool for making decisions about real students.

## 1. Import libraries
Run cells from top to bottom using the ▶ button beside each cell, or choose **Runtime → Run all** in Google Colab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully.")

## 2. Load the dataset
Upload `student_scores_synthetic.csv` to the Colab Files panel, or use the fallback below to generate the same sample dataset directly in the notebook.

In [ ]:
from pathlib import Path

DATA_FILE = "student_scores_synthetic.csv"

if Path(DATA_FILE).exists():
    df = pd.read_csv(DATA_FILE)
    print(f"Loaded dataset: {DATA_FILE}")
else:
    # Reproducible synthetic sample so the notebook still works if the CSV is not uploaded.
    rng = np.random.default_rng(42)
    n = 120
    study_hours = np.round(rng.uniform(0.5, 8.0, n), 1)
    attendance = rng.integers(50, 101, n)
    previous_score = rng.integers(30, 96, n)
    sleep_hours = np.round(rng.uniform(4.0, 9.0, n), 1)
    noise = rng.normal(0, 6, n)
    exam_score = 5.2*study_hours + 0.22*attendance + 0.25*previous_score + sleep_hours - 12 + noise
    exam_score = np.clip(np.round(exam_score), 0, 100).astype(int)
    df = pd.DataFrame({
        "study_hours": study_hours,
        "attendance_percent": attendance,
        "previous_score": previous_score,
        "sleep_hours": sleep_hours,
        "exam_score": exam_score
    })
    df.to_csv(DATA_FILE, index=False)
    print(f"Generated and saved synthetic dataset: {DATA_FILE}")

df.head()

## 3. Inspect and validate the data

In [ ]:
print("Dataset shape (rows, columns):", df.shape)
print("\nColumn data types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nSummary statistics:")
display(df.describe())

required_columns = [
    "study_hours", "attendance_percent", "previous_score",
    "sleep_hours", "exam_score"
]
missing_columns = set(required_columns) - set(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

df = df.dropna(subset=required_columns).copy()
print("Rows after removing missing values:", len(df))

## 4. Exploratory data analysis (EDA)

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["exam_score"], bins=12, edgecolor="black")
plt.title("Distribution of Exam Scores")
plt.xlabel("Exam score")
plt.ylabel("Number of students")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.regplot(data=df, x="study_hours", y="exam_score", scatter_kws={"alpha": 0.65})
plt.title("Study Hours vs Exam Score")
plt.xlabel("Study hours")
plt.ylabel("Exam score")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
corr = df[required_columns].corr(numeric_only=True)
image = plt.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
plt.colorbar(image, label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
for row in range(len(corr.index)):
    for col in range(len(corr.columns)):
        plt.text(col, row, f"{corr.iloc[row, col]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

**How to interpret the graphs:** describe only what your actual graphs show. Correlation does not prove causation. Because the dataset is synthetic, these patterns are illustrative rather than evidence about real students.

## 5. Prepare features and target

In [ ]:
feature_columns = ["study_hours", "attendance_percent", "previous_score", "sleep_hours"]
X = df[feature_columns]
y = df["exam_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 6. Train the Linear Regression model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Model training completed.")
print("Intercept:", round(float(model.intercept_), 3))
print("\nFeature coefficients:")
for feature, coefficient in zip(feature_columns, model.coef_):
    print(f"{feature}: {coefficient:.3f}")

## 7. Evaluate the model

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  (Mean Absolute Error): {mae:.2f} marks")
print(f"RMSE (Root Mean Squared Error): {rmse:.2f} marks")
print(f"R² score: {r2:.3f}")

print("\nInterpretation:")
print(f"On this test split, predictions differ from actual scores by about {mae:.2f} marks on average (MAE).")
print("R² describes how well predictions explain score variation on this test split; it can be negative.")
print("These metrics are based on synthetic data and should not be presented as real-world model performance.")

## 8. Compare actual and predicted scores

In [ ]:
comparison = pd.DataFrame({
    "Actual score": y_test.to_numpy(),
    "Predicted score": np.round(y_pred, 1)
})
display(comparison.head(10))

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.75)
plt.plot([0, 100], [0, 100], linestyle="--")
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.title("Actual vs Predicted Exam Scores")
plt.xlabel("Actual score")
plt.ylabel("Predicted score")
plt.tight_layout()
plt.show()

## 9. Try a prediction for a new example

In [ ]:
# Change these example values to try a different hypothetical student.
new_student = pd.DataFrame([{
    "study_hours": 4.0,
    "attendance_percent": 85,
    "previous_score": 70,
    "sleep_hours": 7.0
}])

predicted_score = float(model.predict(new_student[feature_columns])[0])
predicted_score = float(np.clip(predicted_score, 0, 100))

print("Input values:")
display(new_student)
print(f"Estimated exam score: {predicted_score:.1f} / 100")
print("This is a model estimate, not a guaranteed result.")

## 10. Save useful outputs

In [ ]:
# Save test predictions for inclusion in your results folder.
comparison.to_csv("prediction_results.csv", index=False)
print("Saved prediction_results.csv")

# Save the trained model so it can be reused later.
import joblib
joblib.dump(model, "student_score_linear_regression.joblib")
print("Saved student_score_linear_regression.joblib")

## 11. Conclusion (edit after running)
This project demonstrates a basic supervised machine-learning workflow: loading data, exploratory analysis, training a Linear Regression model, evaluating predictions, and predicting a score for a hypothetical example. The dataset in this starter notebook is synthetic, so the results demonstrate the workflow only. A real deployment would require an appropriate, ethically collected dataset, validation on representative data, and careful consideration of factors not captured by this simple model.

## 12. Viva quick notes
- **What is regression?** Predicting a numerical value.
- **Why Linear Regression?** It is a simple baseline model for predicting a continuous score.
- **What is X?** Input features: study hours, attendance, previous score, and sleep hours.
- **What is y?** The target column, `exam_score`.
- **Why train-test split?** To evaluate the model on data it did not train on.
- **What is MAE?** Average absolute difference between actual and predicted values.
- **What is RMSE?** Square root of the average squared prediction error; it penalizes large errors more than MAE.
- **What is R²?** A measure of explained variation relative to predicting the test-set mean; it can be negative.
- **Main limitation?** The data is synthetic and the model omits many real-world factors; predictions are not guaranteed.